<a href="https://colab.research.google.com/github/kenton116/artist-habit-analysis/blob/dev/chord_to_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Usage:
    python chord_to_json.py <input.txt>              # ファイルから読み込み
    python chord_to_json.py <input.txt> -o out.json  # 出力先を指定
    python chord_to_json.py --stdin                  # 標準入力から読み込み

Setup:
    pip install google-generativeai
    export GEMINI_API_KEY="your_key_here"
"""

import os
import sys
import json
import argparse
import textwrap
import time
from pathlib import Path

try:
    import google.generativeai as genai
except ImportError:
    sys.exit()

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
SYSTEM_PROMPT = textwrap.dedent("""\
    あなたは音楽理論に詳しいアシスタントです。
    ユーザーが送る「歌詞+コード混在テキスト」を解析し、
    以下のJSONフォーマット仕様に厳密に従ってJSONを出力してください。

    ## 基本方針
    - 出力はJSON単体。説明文・コードブロック記法(```)は不要
    - 判断できない項目は null を入れる。推測で埋めない
    - コード記号は入力テキストの表記をそのまま使用する（正規化しない）
    - セクション名が明記されていない場合は構造から推測して英語で記入する
      （例: intro / verse / pre-chorus / chorus / bridge / solo / outro）

    ## 楽器レイヤー
    - 入力に楽器の指示がある場合のみ記録する（表記例: E.Gt / A.Gt / E.Ba / Dr / Vo）
    - "All" と書かれている場合はそれまでに登場した主要楽器をすべて列挙する
    - 指示がない箇所は null

    ## コード長さ
    - "----" のようなリズム表記がある場合: "-" 1つ = 8分音符 = 0.5拍
    - ない場合: 一つ前のコード進行の表記を継承。
    - N.C. は chord_symbol = "N.C." として記録

    ## シンコペーション（小節跨ぎ）
    - 前の小節末尾（例: beat_position=4.5）に 0.5拍分のエントリを置く
    - 次の小節頭に同じコードを再エントリして残り時間を duration_beats に記録

    ## 転調
    - 曲全体のデフォルトは song_meta の key_root / key_mode に記録
    - 転調するセクションのみ section_key_root / section_key_mode に上書き値を記録
    - 転調なしのセクションは null

    ## 繰り返し
    - "×2" "D.C." などがある場合は repeat: true にしてコードを展開して記録

    ## statistics
    - 全フィールドを空値（null または {}）のまま出力すること。計算しない

    ## 出力JSONスキーマ
    {
      "song_meta": {
        "title": str,
        "artist": str,
        "composer": str | null,
        "lyricist": str | null,
        "album": str | null,
        "year": int | null,
        "bpm": int | null,
        "time_signature": str,   // 例: "4/4"
        "key_root": str,         // 例: "G", "Bb", "F#"
        "key_mode": "major" | "minor"
      },
      "sections": [
        {
          "section_id": str,          // 例: "S01"
          "section_type": str,        // intro|verse|pre-chorus|chorus|bridge|solo|outro|interlude
          "section_label": str | null,
          "repeat": bool,
          "instrument_layer": list[str] | null,
          "measure_start": int,
          "measure_end": int,
          "section_key_root": str | null,
          "section_key_mode": str | null,
          "chords": [
            {
              "measure_number": int,
              "beat_position": float,   // 1.0始まり、0.5刻み
              "chord_symbol": str,
              "duration_beats": float,
              "lyric_syllable": str | null
            }
          ]
        }
      ],
      "statistics": {
        "total_measures": null,
        "total_chord_events": null,
        "chord_frequency": {},
        "degree_frequency": {},
        "transition_matrix": {},
        "most_common_progression": [],
        "unique_chords": [],
        "unique_degrees": [],
        "secondary_dominants": [],
        "borrowed_chords": [],
        "cadence_types": {"authentic": null, "half": null, "plagal": null, "deceptive": null}
      }
    }
""")


# ─── メイン処理 ──────────────────────────────────────────────

def load_api_key() -> str:
    key = os.environ.get("GEMINI_API_KEY", "")
    if not key:
        sys.exit("GEMINI_API_KEY is not found")
    return key


def call_gemini(chord_text: str, model_name: str = "gemini-2.0-flash", retries: int = 3) -> str:
    """Gemini API を呼び出してレスポンステキストを返す"""
    genai.configure(api_key=load_api_key())
    model = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=SYSTEM_PROMPT,
    )

    for attempt in range(1, retries + 1):
        try:
            print(f"INFO: (試行 {attempt}/{retries})")
            response = model.generate_content(chord_text)
            return response.text
        except Exception as e:
            print(f"ERROR: {e}")
            if attempt < retries:
                wait = 5 * attempt
                print(f"{wait}秒後にリトライします...")
                time.sleep(wait)
            else:
                sys.exit(f"ERROR: {e}")


def parse_json_response(raw: str) -> dict:
    text = raw.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        text = "\n".join(
            line for line in lines
            if not line.strip().startswith("```")
        ).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        debug_path = Path("debug_raw_response.txt")
        debug_path.write_text(raw, encoding="utf-8")
        sys.exit(
            f"ERROR: {e}\n"
            f"        生レスポンスを {debug_path} に保存しました。"
        )


def resolve_output_path(input_path: Path | None, specified: str | None) -> Path:
    if specified:
        return Path(specified)
    if input_path:
        return input_path.with_suffix(".json")
    return Path("output.json")


def main():
    parser = argparse.ArgumentParser(
        description="コード+歌詞テキストを Gemini API で JSON に変換する"
    )
    parser.add_argument("input", nargs="?", help="入力テキストファイルのパス")
    parser.add_argument("-o", "--output", help="出力JSONファイルのパス（省略時は入力と同名.json）")
    parser.add_argument("--stdin", action="store_true", help="標準入力からテキストを読み込む")
    parser.add_argument("--model", default="gemini-2.0-flash", help="使用するモデル名（デフォルト: gemini-2.0-flash）")
    parser.add_argument("--dry-run", action="store_true", help="APIを呼ばずプロンプトだけ表示して終了")
    args = parser.parse_args()

    # ── 入力テキスト読み込み ──
    input_path = None
    if args.stdin:
        print("[INFO] 標準入力からテキストを読み込みます（Ctrl+D で終了）")
        chord_text = sys.stdin.read()
    elif args.input:
        input_path = Path(args.input)
        if not input_path.exists():
            sys.exit(f"[ERROR] ファイルが見つかりません: {input_path}")
        chord_text = input_path.read_text(encoding="utf-8")
        print(f"[INFO] 読み込み完了: {input_path} ({len(chord_text)} 文字)")
    else:
        parser.print_help()
        sys.exit(1)

    # ── ドライラン ──
    if args.dry_run:
        print("\n===== SYSTEM PROMPT =====")
        print(SYSTEM_PROMPT)
        print("\n===== USER INPUT (先頭200文字) =====")
        print(chord_text[:200])
        sys.exit(0)

    # ── API 呼び出し ──
    raw_response = call_gemini(chord_text, model_name=args.model)

    # ── JSON パース ──
    result = parse_json_response(raw_response)

    # ── 出力 ──
    output_path = resolve_output_path(input_path, args.output)
    output_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"[OK] 保存完了: {output_path}")

    # ── サマリー表示 ──
    meta = result.get("song_meta", {})
    sections = result.get("sections", [])
    total_chords = sum(len(s.get("chords", [])) for s in sections)
    print(f"     タイトル  : {meta.get('title')}")
    print(f"     アーティスト: {meta.get('artist')}")
    print(f"     Key       : {meta.get('key_root')} {meta.get('key_mode')}")
    print(f"     セクション数: {len(sections)}")
    print(f"     コードイベント数: {total_chords}")


if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
usage: colab_kernel_launcher.py [-h] [-o OUTPUT] [--stdin] [--model MODEL]
                                [--dry-run]
                                [input]
colab_kernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
